# Lab 2.1: Feature Extraction & Word Representations

This notebook demonstrates classical and distributed text representations on a simple sample corpus:
1. **CountVectorizer (`cv`)** — Bag-of-Words (BoW)
2. **TF-IDF Vectorizer (`tfidf`)** — Term Frequency - Inverse Document Frequency
3. **Co-occurrence Matrix (`co.m`)** — Word-Word context co-occurrence counts
4. **Word2Vec** — Distributed neural embeddings using `gensim`
5. **GloVe** — Global co-occurrence factorization using weighted least squares

**Libraries**: `scikit-learn`, `nltk`, `gensim`, `pandas`, `numpy`.

In [7]:
# Core libraries
import numpy as np
import pandas as pd
from IPython.display import display

# NLP & ML Libraries
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

# Ensure NLTK tokenizer models are available
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Sample corpus for demonstration
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the lazy dog barks at the quick brown fox",
    "the fox and the dog are friends"
]

print("Sample Corpus:")
for i, doc in enumerate(corpus):
    print(f"Document {i+1}: {doc}")

Sample Corpus:
Document 1: the quick brown fox jumps over the lazy dog
Document 2: the lazy dog barks at the quick brown fox
Document 3: the fox and the dog are friends


In [8]:
cv = CountVectorizer()
cv_matrix = cv.fit_transform(corpus)

df_cv = pd.DataFrame(
    cv_matrix.toarray(),
    columns=cv.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("--- CountVectorizer (BoW) Matrix ---")
display(df_cv)

--- CountVectorizer (BoW) Matrix ---


,and,are,at,barks,brown,dog,fox,friends,jumps,lazy,over,quick,the
Doc 1,0,0,0,0,1,1,1,0,1,1,1,1,2
Doc 2,0,0,1,1,1,1,1,0,0,1,0,1,2
Doc 3,1,1,0,0,0,1,1,1,0,0,0,0,2


In [9]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("--- TF-IDF Matrix ---")
display(df_tfidf)

--- TF-IDF Matrix ---


,and,are,at,barks,brown,dog,fox,friends,jumps,lazy,over,quick,the
Doc 1,0.000,0.000,0.000,0.000,0.315,0.245,0.245,0.000,0.414,0.315,0.414,0.315,0.489
Doc 2,0.000,0.000,0.414,0.414,0.315,0.245,0.245,0.000,0.000,0.315,0.000,0.315,0.489
Doc 3,0.443,0.443,0.000,0.000,0.000,0.262,0.262,0.443,0.000,0.000,0.000,0.000,0.523


In [10]:

tokenized_docs = [word_tokenize(doc.lower()) for doc in corpus]

vocab = sorted(list(set(w for doc in tokenized_docs for w in doc)))
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

window_size = 2
co_matrix = np.zeros((V, V), dtype=int)

for doc in tokenized_docs:
    for i, word in enumerate(doc):
        start = max(0, i - window_size)
        end = min(len(doc), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                co_matrix[w2i[word]][w2i[doc[j]]] += 1

df_cooc = pd.DataFrame(co_matrix, index=vocab, columns=vocab)
print(f"--- Word-Word Co-occurrence Matrix (Window Size = {window_size}) ---")
display(df_cooc)

--- Word-Word Co-occurrence Matrix (Window Size = 2) ---


,and,are,at,barks,brown,dog,fox,friends,jumps,lazy,over,quick,the
and,0,0,0,0,0,1,1,0,0,0,0,0,2
are,0,0,0,0,0,1,0,1,0,0,0,0,1
at,0,0,0,1,0,1,0,0,0,0,0,1,1
barks,0,0,1,0,0,1,0,0,0,1,0,0,1
brown,0,0,0,0,0,0,2,0,1,0,0,2,2
dog,1,1,1,1,0,0,0,1,0,2,0,0,3
fox,1,0,0,0,2,0,0,0,1,0,1,2,2
friends,0,1,0,0,0,1,0,0,0,0,0,0,0
jumps,0,0,0,0,1,0,1,0,0,0,1,0,1
lazy,0,0,0,1,0,2,0,0,0,0,1,0,2


In [11]:

training_data = tokenized_docs * 30

w2v_model = Word2Vec(
    sentences=training_data,
    vector_size=10,
    window=2,
    min_count=1,
    sg=1,
    epochs=50,
    seed=42
)

print("--- Word2Vec Model ---")
print("Embedding vector for 'fox':\n", w2v_model.wv['fox'].round(3))
print("\nCosine similarity between 'fox' and 'dog':", round(float(w2v_model.wv.similarity('fox', 'dog')), 3))
print("Most similar words to 'fox':", [w for w, s in w2v_model.wv.most_similar('fox', topn=3)])

--- Word2Vec Model ---
Embedding vector for 'fox':
 [-0.14   0.453  0.115  0.215  0.753  0.501 -0.614  0.315  0.122 -0.151]

Cosine similarity between 'fox' and 'dog': 0.981
Most similar words to 'fox': ['lazy', 'jumps', 'quick']


In [12]:
def glove_weight(x, x_max=10.0, alpha=0.75):
    return (x / x_max) ** alpha if x < x_max else 1.0

dim = 5
rng = np.random.default_rng(42)
W = rng.normal(0, 0.1, (V, dim))
W_tilde = rng.normal(0, 0.1, (V, dim))
b = np.zeros(V)
b_tilde = np.zeros(V)

# Train GloVe with simple gradient descent
lr = 0.05
for epoch in range(150):
    for i in range(V):
        for j in range(V):
            x_ij = co_matrix[i, j]
            if x_ij > 0:
                weight = glove_weight(x_ij)
                diff = np.dot(W[i], W_tilde[j]) + b[i] + b_tilde[j] - np.log(x_ij)
                grad = 2.0 * weight * diff
                
                W[i] -= lr * grad * W_tilde[j]
                W_tilde[j] -= lr * grad * W[i]
                b[i] -= lr * grad
                b_tilde[j] -= lr * grad

# Final GloVe representations (sum of word and context vectors)
glove_vectors = W + W_tilde

# Display learned GloVe vectors
df_glove = pd.DataFrame(
    glove_vectors.round(3),
    index=vocab,
    columns=[f"dim_{k+1}" for k in range(dim)]
)
print("--- GloVe Learned Word Vectors ---")
display(df_glove.head(6))

# Measure GloVe similarity between 'fox' and 'dog' using sklearn cosine_similarity
fox_vec = glove_vectors[w2i['fox']].reshape(1, -1)
dog_vec = glove_vectors[w2i['dog']].reshape(1, -1)
sim_fox_dog = cosine_similarity(fox_vec, dog_vec)[0, 0]
print(f"\nGloVe cosine similarity between 'fox' and 'dog': {sim_fox_dog:.3f}")

--- GloVe Learned Word Vectors ---


,dim_1,dim_2,dim_3,dim_4,dim_5
and,-0.021,0.065,0.092,0.162,-0.354
are,-0.062,-0.239,-0.220,0.126,0.015
at,0.395,-0.221,-0.153,0.404,-0.021
barks,0.310,-0.306,-0.180,0.365,-0.031
brown,-0.051,-0.189,0.185,-0.079,0.138
dog,-0.356,0.564,0.128,-0.406,-0.119



GloVe cosine similarity between 'fox' and 'dog': -0.254
